# Autoencoder para detecção de spam em SMS

**Disciplina:** Redes Neurais  
**Integrantes:** preencher  
**Turma:** preencher

O sistema recebe uma mensagem SMS e indica se ela parece normal (`ham`) ou suspeita de spam. O autoencoder aprende a reconstruir mensagens legítimas; erros de reconstrução acima de um limite são tratados como anomalias.

> A base escolhida contém mensagens em inglês. O protótipo não foi validado para português.


## 1. Bibliotecas e carregamento dos dados

Usaremos a SMS Spam Collection, do UCI Machine Learning Repository (5.574 mensagens rotuladas como `ham` ou `spam`).

In [ ]:
%pip -q install tensorflow scikit-learn pandas matplotlib requests

import io, zipfile, time, random, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, ConfusionMatrixDisplay, classification_report
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
response = requests.get(url, timeout=60); response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    filename = next(n for n in z.namelist() if n.endswith('SMSSpamCollection'))
    raw = z.read(filename).decode('utf-8', errors='replace')
rows = [line.split('\t', 1) for line in raw.splitlines() if '\t' in line]
df = pd.DataFrame(rows, columns=['label','text'])
df['label'] = df['label'].str.strip(); df['text'] = df['text'].str.strip()
df = df[df['label'].isin(['ham','spam']) & df['text'].ne('')].copy()
# Remove textos que aparecem com rótulos conflitantes e duplicatas.
ambiguous = df.groupby('text')['label'].nunique()
df = df[~df['text'].isin(ambiguous[ambiguous > 1].index)].drop_duplicates('text').reset_index(drop=True)
print('Mensagens após limpeza:', len(df))
display(df['label'].value_counts().rename(index={'ham':'Normal (ham)','spam':'Spam'}).to_frame('Quantidade'))

## 2. Divisão dos dados

Separação estratificada: 70% treino, 15% validação e 15% teste. O vocabulário TF-IDF será ajustado somente com o treino, para evitar vazamento de dados.

In [ ]:
X = df['text']; y = df['label']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(f'Treino: {len(X_train)} ({len(X_train)/len(df):.1%})')
print(f'Validação: {len(X_val)} ({len(X_val)/len(df):.1%})')
print(f'Teste: {len(X_test)} ({len(X_test)/len(df):.1%})')
display(pd.DataFrame({'treino':y_train.value_counts(normalize=True),'validação':y_val.value_counts(normalize=True),'teste':y_test.value_counts(normalize=True)}))

## 3. Representação dos textos

TF-IDF converte cada mensagem em um vetor numérico: cada posição representa um termo ou combinação de termos e seu peso indica importância relativa. Não é um embedding contextual e não compreende plenamente o significado da frase.

In [ ]:
vectorizer = TfidfVectorizer(lowercase=True, strip_accents='unicode', max_features=3000, ngram_range=(1,2), min_df=2, max_df=0.98, norm='l2', dtype=np.float32)X_train_tfidf = vectorizer.fit_transform(X_train)X_val_tfidf = vectorizer.transform(X_val)X_test_tfidf = vectorizer.transform(X_test)print('Número de características:', X_train_tfidf.shape[1])print('Exemplo de mensagem:', X_train.iloc[0])print('Vetor TF-IDF (valores não zero):', X_train_tfidf[0].data[:10])

## 4. Baseline: regressão logística

A regressão logística é uma referência tradicional e supervisionada. Ela aprende diretamente com os rótulos `ham` e `spam`; `class_weight='balanced'` ajuda a compensar o desbalanceamento.

In [ ]:
baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
baseline.fit(X_train_tfidf, y_train)
pred_baseline = baseline.predict(X_test_tfidf)
score_baseline = baseline.predict_proba(X_test_tfidf)[:, list(baseline.classes_).index('spam')]
print('Baseline treinada.')

## 5. Arquitetura do autoencoderO encoder comprime o vetor de entrada em uma representação latente; o decoder tenta reconstruir o vetor original. O treino utiliza somente mensagens normais (`ham`).**Problema identificado na primeira versão:** os vetores TF-IDF são esparsos, com muitas posições iguais a zero. A MSE comum pode ser dominada por esses zeros e permitir que a rede reconstrua quase tudo como zero. Nesta versão usamos uma **MSE ponderada**: peso 1 para características presentes e peso 0,005 para posições ausentes.A arquitetura agora é 256 → 128 → 64 → 128 → 256, com gargalo de 64 dimensões. Isso continua sendo detecção de anomalias one-class/semi-supervisionada, não classificação supervisionada pelo autoencoder.

In [ ]:
Xtr = X_train_tfidf.toarray().astype(np.float32)Xva = X_val_tfidf.toarray().astype(np.float32)Xte = X_test_tfidf.toarray().astype(np.float32)ytr = y_train.to_numpy(); yva = y_val.to_numpy(); yte = y_test.to_numpy()Xtr_ham = Xtr[ytr == 'ham']; Xva_ham = Xva[yva == 'ham']n_features = Xtr.shape[1]# A MSE comum dá peso igual a todos os termos; como há muitos zeros,# a rede pode aprender a reconstruir quase tudo como zero.PESO_CARACTERISTICA_AUSENTE = 0.005def mse_ponderada(y_true, y_pred):    mascara_ativa = tf.cast(y_true > 0.0, tf.float32)    pesos = mascara_ativa + PESO_CARACTERISTICA_AUSENTE * (1.0 - mascara_ativa)    erro_por_amostra = tf.reduce_sum(        pesos * tf.square(y_true - y_pred), axis=1    ) / tf.reduce_sum(pesos, axis=1)    return erro_por_amostrainp = Input(shape=(n_features,), name='vetor_tfidf')x = Dense(256, activation='relu', name='encoder_256')(inp)x = Dense(128, activation='relu', name='encoder_128')(x)latent = Dense(64, activation='relu', name='gargalo')(x)x = Dense(128, activation='relu', name='decoder_128')(latent)x = Dense(256, activation='relu', name='decoder_256')(x)out = Dense(n_features, activation='sigmoid', name='reconstrucao')(x)autoencoder = Model(inp, out, name='autoencoder_ponderado')autoencoder.compile(    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),    loss=mse_ponderada)autoencoder.summary()

## 6. Treinamento e limite de anomaliaReLU ativa as camadas ocultas; sigmoid mantém a reconstrução entre 0 e 1. A perda é uma MSE ponderada: peso 1 para termos presentes e 0,005 para posições ausentes. Adam ajusta os pesos e Early Stopping reduz o risco de sobreajuste.A saída sigmoid reconstrói TF-IDF; não é uma probabilidade de spam. O limite é calculado pelo percentil 95 dos erros das mensagens ham na validação.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1)start = time.time()history = autoencoder.fit(    Xtr_ham, Xtr_ham,    validation_data=(Xva_ham, Xva_ham),    epochs=60, batch_size=32, shuffle=True,    callbacks=[early_stop], verbose=1)training_seconds = time.time() - startprint(f'Épocas executadas: {len(history.history["loss"])} | Tempo: {training_seconds:.1f} s')plt.figure(figsize=(8,4))plt.plot(history.history['loss'], label='Treino')plt.plot(history.history['val_loss'], label='Validação')plt.xlabel('Época'); plt.ylabel('MSE ponderada')plt.title('Perda de reconstrução ponderada')plt.legend(); plt.grid(alpha=.2); plt.show()def reconstruction_error(matrix):    reconstructed = autoencoder.predict(matrix, verbose=0)    active = (matrix > 0.0).astype(np.float32)    weights = active + PESO_CARACTERISTICA_AUSENTE * (1.0 - active)    return np.sum(weights * np.square(matrix - reconstructed), axis=1) / np.sum(weights, axis=1)val_errors_ham = reconstruction_error(Xva_ham)threshold = float(np.percentile(val_errors_ham, 95))val_errors_spam = reconstruction_error(Xva[yva == 'spam'])print(f'Limite (percentil 95 ham na validação): {threshold:.8f}')print(f'Mediana do erro ham na validação: {np.median(val_errors_ham):.8f}')# Diagnóstico visual: os exemplos spam da validação não são usados para definir o limite.plt.figure(figsize=(8,4))plt.hist(val_errors_ham, bins=35, alpha=.65, density=True, label='Normal (ham)')plt.hist(val_errors_spam, bins=25, alpha=.55, density=True, label='Spam (diagnóstico)')plt.axvline(threshold, color='red', linestyle='--', label='Limite P95 ham')plt.xlabel('Erro de reconstrução ponderado')plt.ylabel('Densidade')plt.title('Distribuição do erro na validação')plt.legend(); plt.tight_layout(); plt.show()

## 7. Avaliação no conjunto de teste

Erro de reconstrução acima do limite é marcado como spam. As métricas de precisão, recall e F1 são calculadas para a classe spam. O conjunto de teste só é usado nesta etapa final.

In [ ]:
errors_test = reconstruction_error(Xte)
pred_ae = (errors_test > threshold).astype(int)
y_test_bin = (yte == 'spam').astype(int)
pred_base_bin = (pred_baseline == 'spam').astype(int)
def metrics(name, y_true, pred, scores):
    return {'Modelo':name, 'Acurácia':accuracy_score(y_true,pred), 'Precisão spam':precision_score(y_true,pred,zero_division=0), 'Recall spam':recall_score(y_true,pred,zero_division=0), 'F1 spam':f1_score(y_true,pred,zero_division=0), 'ROC-AUC':roc_auc_score(y_true,scores), 'Average Precision':average_precision_score(y_true,scores)}
results = pd.DataFrame([metrics('Regressão logística',y_test_bin,pred_base_bin,score_baseline), metrics('Autoencoder',y_test_bin,pred_ae,errors_test)]).set_index('Modelo')
display(results.style.format('{:.4f}'))
print('Relatório do autoencoder:')
print(classification_report(y_test_bin,pred_ae,labels=[0,1],target_names=['Normal','Spam'],zero_division=0))
fig, ax = plt.subplots(1,2,figsize=(10,4))
ConfusionMatrixDisplay.from_predictions(y_test_bin,pred_base_bin,display_labels=['Normal','Spam'],ax=ax[0],colorbar=False); ax[0].set_title('Baseline')
ConfusionMatrixDisplay.from_predictions(y_test_bin,pred_ae,display_labels=['Normal','Spam'],ax=ax[1],colorbar=False); ax[1].set_title('Autoencoder')
plt.tight_layout(); plt.show()
print('Diferença F1 (autoencoder - baseline):', round(f1_score(y_test_bin,pred_ae)-f1_score(y_test_bin,pred_base_bin),4))

## 8. Análise de erros e demonstração

Falso positivo: mensagem legítima marcada como spam. Falso negativo: spam classificado como normal. Analise os exemplos abaixo para discutir causas possíveis.

In [ ]:
error_table = pd.DataFrame({'mensagem':X_test.to_numpy(),'esperado':yte,'previsto':np.where(pred_ae==1,'spam','ham'),'erro':errors_test})
print('Falsos positivos:')
display(error_table[(error_table.esperado=='ham') & (error_table.previsto=='spam')].head(3))
print('Falsos negativos:')
display(error_table[(error_table.esperado=='spam') & (error_table.previsto=='ham')].head(3))
def analisar_mensagem(texto):
    if not isinstance(texto,str) or not texto.strip(): raise ValueError('Digite uma mensagem não vazia.')
    vetor = vectorizer.transform([texto]).toarray().astype(np.float32)
    erro = float(reconstruction_error(vetor)[0])
    return {'mensagem':texto,'classificação':'SPAM (suspeita)' if erro > threshold else 'NORMAL (ham)','erro':erro,'limite':threshold,'observação':'O erro não é uma probabilidade de spam.'}
display(analisar_mensagem('Congratulations! You have won a free prize. Claim now!'))
nova_mensagem = input('Digite uma mensagem em inglês para analisar: ')
display(analisar_mensagem(nova_mensagem))

In [ ]:
# Exemplos completos para o slide 13.# Execute depois da célula de análise dos erros.pd.set_option("display.max_colwidth", None)error_table["correto"] = error_table["esperado"] == error_table["previsto"]print("EXEMPLO DE ACERTO")acertos = error_table[error_table["correto"]]if not acertos.empty:    ex = acertos.iloc[0]    print("Mensagem:", ex["mensagem"])    print("Esperado:", ex["esperado"])    print("Previsto:", ex["previsto"])else:    print("Não houve acertos nesta execução.")print("\nEXEMPLO DE FALSO NEGATIVO")falsos_negativos = error_table[(error_table["esperado"] == "spam") & (error_table["previsto"] == "ham")]if not falsos_negativos.empty:    ex = falsos_negativos.iloc[0]    print("Mensagem:", ex["mensagem"])    print("Esperado:", ex["esperado"])    print("Previsto:", ex["previsto"])    print("Erro ponderado:", f"{ex['erro']:.8f}")else:    print("Não houve falsos negativos nesta execução.")print("\nEXEMPLO DE FALSO POSITIVO")falsos_positivos = error_table[(error_table["esperado"] == "ham") & (error_table["previsto"] == "spam")]if not falsos_positivos.empty:    ex = falsos_positivos.iloc[0]    print("Mensagem:", ex["mensagem"])    print("Esperado:", ex["esperado"])    print("Previsto:", ex["previsto"])else:    print("Não houve falsos positivos nesta execução.")

## 9. Limitações, ética e referências- A base contém SMS em inglês e é histórica; não representa todo o spam atual nem mensagens em português.- TF-IDF não compreende contexto ou intenção; mensagens legítimas incomuns podem gerar falsos positivos.- A ponderação da reconstrução é uma melhoria experimental, não uma garantia de detecção.- O limite é uma heurística calibrada na validação. O teste permanece separado.- Mensagens podem conter dados pessoais: minimizar coleta, proteger os dados e respeitar a legislação aplicável.**Declaração de IA:** inteligência artificial generativa foi usada como apoio à organização, explicação dos conceitos e elaboração inicial do notebook. A equipe deve revisar, executar, adaptar e compreender o código.**Referências:** UCI Machine Learning Repository, SMS Spam Collection (https://archive.ics.uci.edu/dataset/228/sms); Pedregosa et al., *Scikit-learn: Machine Learning in Python*, JMLR, 2011; Abadi et al., *TensorFlow: A System for Large-Scale Machine Learning*, 2016.